This notebook covers steps 7–12 of the Ubuntu Dialogue Corpus workflow.

It rebuilds steps 1–6 in the same kernel rather than reading an intermediate pickle checkpoint. After the human normalization, structural anonymization, lexicon/slang matching, glued-term matching, residual-vocabulary analysis, and human-validated overlays are complete, the immediate next stage is sentiment analysis.


In [ ]:
# Imports, repository paths, and adaptive local parallelism
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pipeline').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pipeline').is_dir():
    raise FileNotFoundError('Run this notebook from the repository root or notebooks directory')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from pipeline.parallel_execution import map_rows, recommended_workers

# Set an integer to cap workers, or leave None for automatic CPU-based selection.
MAX_WORKERS = None

def workers_for(row_count):
    return recommended_workers(row_count, max_workers=MAX_WORKERS)

pd.set_option('display.max_colwidth', None)


In [ ]:
pd.set_option('display.max_colwidth', None)  # Show full content of columns

In [ ]:
# Rebuild steps 1–6 in this kernel; no intermediate pickle is written.
steps_1_6 = PROJECT_ROOT / 'notebooks' / 'Ubuntu_Project_Steps1-6.ipynb'
get_ipython().run_line_magic('run', str(steps_1_6))
print(f'Prepared {len(df):,} rows and {len(df.columns)} columns from steps 1–6')


In [ ]:
(df['word_count'] == 0).sum() #check

In [ ]:
df[(df['word_count'] == 0)].head(5)[['date','from','to','text_length','word_count','text']]

In [ ]:
#remove these
df = df[df['word_count'] != 0].reset_index(drop=True)

In [ ]:
(df['text_length'] == 0).sum() #check

In [ ]:
print("blank 'from':", (df['from'] == '').sum())
print("blank 'to':", (df['to'] == '').sum())

In [ ]:
df['from'].astype(str).str.strip().eq('').sum()

In [ ]:
df['to'].astype(str).str.strip().eq('').sum()

In [ ]:
#Check if there's any conversations with more than 2 users
participant_counts = df.groupby('conversation_id')['from'].nunique()
multi_party_ids = participant_counts[participant_counts > 2].index

print("conversations with 3+ users:", len(multi_party_ids))
print("as % of all conversations:", round(len(multi_party_ids) / df['conversation_id'].nunique() * 100, 2))
print("rows affected:", df['conversation_id'].isin(multi_party_ids).sum())

In [ ]:
#Create a new variable that measures response times between users only, not within
prev_from = df.groupby('conversation_id')['from'].shift()
speaker_changed = df['from'] != prev_from

df['response_gap_mins_between_speakers'] = df['response_gap_mins'].where(speaker_changed)

print("dtype:", df['response_gap_mins_between_speakers'].dtype)
print("NaN in response_gap_mins:", df['response_gap_mins'].isna().sum())
print("NaN in response_gap_mins_between_speakers:", df['response_gap_mins_between_speakers'].isna().sum())

---
Evidently, there are 547 non-NaN yet empty text strings, likely filled with "tabs" or other whitespace characters. We have removed these from the dataset. 

Other thoughts: We were seeing many zeros in response_gap_mins... currently, it only checks the immediate prior message. But in reality, there should also be a subset where it is the response_gap_mins between users only. So it only looks at messages where turn count +=1 and is_op != is_op. otherwise ignores. This is important because a user may send multiple messages in quick succession, underrepresenting the time taken to actually respond to the other user, as the user is just replying to himself/herself. 

---
Step 7: Deep Exploration 📊 
- groupby() summaries
- value_counts() distributions
- pivot_table() cross-tabs
- Check for outliers
- Identify patterns

Grouping by: Year, Month, User, Conversation_ID, days_until_release / since_release buckets, is_op, text_length and word_count buckets, user_message_counts, response_gap_mins buckets

In [ ]:
#Group By
# Year / Month
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

# release-distance buckets — weekly bins
release_bins = [-1, 7, 14, 21, 28, 42, 56, float('inf')]
release_labels = ['0-1wk', '1-2wk', '2-3wk', '3-4wk', '4-6wk', '6-8wk', '8wk+']
df['days_since_release_bucket'] = pd.cut(df['days_since_release'], bins=release_bins, labels=release_labels)
df['days_until_release_bucket'] = pd.cut(df['days_until_release'], bins=release_bins, labels=release_labels)

# message length buckets
df['text_length_bucket'] = pd.cut(df['text_length'], bins=[0, 20, 50, 100, 200, float('inf')],
                                    labels=['very_short', 'short', 'medium', 'long', 'very_long'])
df['word_count_bucket'] = pd.cut(df['word_count'], bins=[0, 5, 10, 20, 40, float('inf')],
                                   labels=['very_short', 'short', 'medium', 'long', 'very_long'])

# user-tiers based on messages sent per user per month
monthly_counts = (
    df.groupby(['from', 'year', 'month'], observed=True)
    .size()
    .rename('user_messages_this_month')
    .reset_index()
)
df = df.merge(monthly_counts, on=['from', 'year', 'month'], how='left')

# response latency buckets
df['response_gap_bucket'] = pd.cut(df['response_gap_mins_between_speakers'], bins=[-1, 5, 30, 120, 1440, float('inf')],
                                     labels=['<5min', '5-30min', '30min-2hr', '2hr-1day', '1day+'])

In [ ]:
df['user_messages_this_month'].describe() #check

In [ ]:
#Value Count Distributions, Group By
df['user_tier'] = pd.cut(df['user_messages_this_month'], #User Tier Categorical Proportions
                          bins=[0, 10, 50, 150, 500, float('inf')],
                          labels=['occasional', 'regular', 'active', 'frequent', 'power_user'])
print(df['user_tier'].value_counts(normalize=True))

---
User_tiers -- Earlier operational definition of this variable utilized mere cumulative message count per user. The engineered user tiers feature should instead examine messages sent per month or similar, not simply per conversation or cumulative. Legacy users are persistent, but other users may have been active in earlier years (high cumulative count), but hardly post anymore, and that should be reflected. 

---

In [ ]:
df['days_since_release_bucket'].value_counts(dropna=False).sort_index() #days_since_release

In [ ]:
df['days_until_release_bucket'].value_counts().sort_index() #days_until_release

In [ ]:
df.groupby('days_since_release_bucket', observed=True, dropna=False)['is_op'].mean() #portion of messages sent by OP by time since last release

In [ ]:
df.groupby('days_until_release_bucket', observed=True)['is_op'].mean() #portion of messages sent by OP by time until next release

In [ ]:
df.groupby('days_since_release_bucket', observed=True, dropna=False)[['text_length', 'word_count']].mean() #text length/word count by days since release

In [ ]:
df.groupby('days_until_release_bucket', observed=True)[['text_length', 'word_count']].mean() #text length/word count by days until release

In [ ]:
df.groupby('days_since_release_bucket', observed=True, dropna=False)['response_gap_mins_between_speakers'].median() #response gap in minutes by days since release

In [ ]:
df.groupby('days_until_release_bucket', observed=True)['response_gap_mins_between_speakers'].median() #response gap in minutes by days until release

In [ ]:
df.groupby('user_tier', observed=True)['response_gap_mins_between_speakers'].median()

In [ ]:
df.groupby('user_tier', observed=True)['is_op'].mean()

In [ ]:
#Pivot Table
pd.pivot_table(df, index='days_since_release_bucket', columns='user_tier', values='conversation_id', aggfunc='count', observed=True)

In [ ]:
#Examine the new numeric columns with describe, determine patterns, outliers, and need for distribution transformations
df.drop(columns=['date', 'conversation_id']).describe().apply(lambda x: x.apply('{:.2f}'.format)) #describe with hundredths precision

In [ ]:
#Introduce key identifier per message and place conversation_id second
df['message_id'] = pd.array(range(1, len(df) + 1), dtype='uint32')

cols = ['message_id', 'conversation_id', 'date'] + [c for c in df.columns if c not in ['message_id', 'conversation_id', 'date']]
df = df[cols]

In [ ]:
df.sample(3)

---
EDA Notes: 

Data Health Check:
- check known NaN (present only in days_since_release and response_gap_mins) as a percentage of total rows. 
- identify any zero-variance/constant columns, which present issues downstream

Univariate:
1. Continuous, Numeric Variables: First we do single variable distributions and see about outliers, and whether transformations are necessary. 
- Calculate skewness and kurtosis, identify outliers via Z-score or IQR. 
- Visuals: box/whisker, histogram... Q-Q, residuals plot, density plot (consider natural floor and ceiling in data, use Subject Matter Expertise)
2. Binary, Categorical:
- Class imbalances, concentration spikes
- pie chart, bar chart, or similar.


Multivariate: Relational
- correlation matrix, identify variables that correlate +/-0.5
- Visuals: correlation heatmap, scatter matrix, line charts (time, distance, dosage, temperature, position)
- We see if any variables pose risks for multicollinearity (high correlation + VIF), as well as potential need for interaction terms or PCA/similar. 
- note: always prevent dummy variable bias by leaving out one category AKA the base case (often largest group size, or control/zero-state)

In [ ]:
#Health Check
missing_pct = (df.isna().sum() / len(df) * 100).round(2)
print(missing_pct[missing_pct > 0])

constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
print("Constant columns:", constant_cols)

In [ ]:
#Univariate:
continuous_cols = ['turn_count', 'response_gap_mins_between_speakers', 'user_message_count',
                    'text_length', 'word_count', 'days_since_release', 'days_until_release']

In [ ]:
# skew + kurtosis
skew_kurt = pd.DataFrame({
    'skew': df[continuous_cols].skew(),
    'kurtosis': df[continuous_cols].kurt()
})
print(skew_kurt) 
#kurtosis of +/- 7 and skewness of +/- 2 considered problematic as violating laws of normality

In [ ]:
# outliers: IQR and Z-score side by side
def iqr_outlier_count(series):
    s = series.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lower) | (s > upper)).sum())

def zscore_outlier_count(series, threshold=3):
    s = series.dropna()
    z = (s - s.mean()) / s.std()
    return int((z.abs() > threshold).sum())

outlier_summary = pd.DataFrame({
    'iqr_outliers': {c: iqr_outlier_count(df[c]) for c in continuous_cols},
    'zscore_outliers': {c: zscore_outlier_count(df[c]) for c in continuous_cols},
})

outlier_summary['iqr_pct'] = (outlier_summary['iqr_outliers'] / len(df) * 100).round(2)
outlier_summary['zscore_pct'] = (outlier_summary['zscore_outliers'] / len(df) * 100).round(2)
print(outlier_summary)
#IQR more trustworthy for heavy-skewed data as median is more robust

In [ ]:
#Examine outliers
def iqr_bounds(series):
    s = series.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

outlier_flags = pd.DataFrame(index=df.index)
for col in continuous_cols:
    lower, upper = iqr_bounds(df[col])
    outlier_flags[col] = (df[col] < lower) | (df[col] > upper)

outlier_column_count = outlier_flags.sum(axis=1)
is_outlier_any = outlier_column_count > 0

print("outliers per column:")
print(outlier_flags.sum().sort_values(ascending=False))

print("\nhow many columns flaged in the same row:")
print(outlier_column_count.value_counts().sort_index())

print(f"\nrows flagged anywhere: {is_outlier_any.sum():,} ({is_outlier_any.mean()*100:.1f}%)")

---
Given the disparity of the data, it may be appropriate to apply cluster Analysis (e.g., K-Means) to find hidden, unlabeled groupings or segmentations in the data without a pre-defined target. Additionally, k-Nearest Neighbors (k-NN) can help predict a label or value for new data by looking at the k closest labeled examples in a reference set.

---

In [ ]:
# visuals - accounting for skewness
# histogram, box/whisker, density, Q-Q — one 2x2 grid per continuous column
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

SKEW_MODERATE = 2   # log1p above this
SKEW_SEVERE = 5      # Yeo-Johnson above this
transform_registry = {}
transformed_data = {}

for col in continuous_cols:
    data = df[col].dropna()
    raw_skew = data.skew()
    if raw_skew > SKEW_SEVERE:
        vals, lam = stats.yeojohnson(data.to_numpy())
        transformed = pd.Series(vals, index=data.index)
        transform_name = 'yeojohnson'
        label = f'yeojohnson({col})'
        transform_note = f'Yeo-Johnson (lambda={lam:.3f})'
    elif raw_skew > SKEW_MODERATE:
        transformed = np.log1p(data)
        lam = None
        transform_name = 'log1p'
        label = f'log1p({col})'
        transform_note = 'log1p'
    else:
        transformed = data
        lam = None
        transform_name = 'none'
        label = col
        transform_note = 'none'

    transform_registry[col] = {'transform': transform_name, 'lambda': lam, 'raw_skew': raw_skew}
    transformed_data[col] = transformed.reindex(df.index)
    plot_data = transformed

    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    fig.suptitle(f'{col}  (raw skew={raw_skew:.2f}, transform={transform_note})')
    sns.histplot(plot_data, ax=axes[0, 0], color='#4C72B0')
    axes[0, 0].set_title('Histogram'); axes[0, 0].set_xlabel(label)
    sns.boxplot(x=plot_data, ax=axes[0, 1], color='#4C72B0')
    axes[0, 1].set_title('Box/Whisker'); axes[0, 1].set_xlabel(label)
    sns.kdeplot(plot_data, ax=axes[1, 0], color='#4C72B0', fill=True)
    axes[1, 0].set_title('Density'); axes[1, 0].set_xlabel(label)
    stats.probplot(plot_data, dist='norm', plot=axes[1, 1])
    axes[1, 1].set_title(f'Q-Q Plot ({label})')
    plt.tight_layout()
    plt.show()

df_transformed = pd.DataFrame(transformed_data)
print(pd.DataFrame(transform_registry).T)

In [ ]:
#Univariate Categorical - Claude
bucket_cols = ['user_tier', 'days_since_release_bucket', 'days_until_release_bucket',
               'text_length_bucket', 'word_count_bucket', 'response_gap_bucket']
categorical_cols = ['is_op'] + bucket_cols

for col in categorical_cols:
    counts = df[col].value_counts(dropna=False)
    pct = (counts / len(df) * 100).round(2)
    print(f"\n{col} -> {df[col].nunique(dropna=False)} levels")
    print(pd.DataFrame({'count': counts, 'pct': pct}))

# binary categorical
fig, ax = plt.subplots(figsize=(4, 4))
df['is_op'].value_counts().plot.pie(autopct='%1.1f%%', ax=ax, colors=['#4C72B0', '#DD8452'])
ax.set_ylabel('')
ax.set_title('is_op')
plt.show()

# multi-level categorical
for col in bucket_cols:
    fig, ax = plt.subplots(figsize=(6, 4))
    order = df[col].value_counts(dropna=True).index  # NaN excluded here
    sns.countplot(data=df, x=col, order=order, ax=ax, color='#4C72B0')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
#Multivariate - Relational:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# correlations
corr_matrix = df_transformed.corr()
print(corr_matrix.round(2))

# flag pairs at |r| >= 0.5
strong_pairs = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack().reset_index()
)
strong_pairs.columns = ['var_1', 'var_2', 'corr']
strong_pairs = strong_pairs[strong_pairs['corr'].abs() >= 0.5].sort_values('corr', key=abs, ascending=False)
print(strong_pairs)

# correlation heatmap - diverging colormap, centered at 0
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, ax=ax)
plt.tight_layout()
plt.show()

# scatter matrix on a sample - full 8.5M rows would overplot into a solid blob
sample = df_transformed.dropna().sample(n=10_000, random_state=42)
pd.plotting.scatter_matrix(sample, figsize=(12, 12), diagonal='kde',
                            color='#4C72B0', alpha=0.3, s=5)
plt.tight_layout()
plt.show()

# line chart - message volume over time
monthly = df.groupby(['year', 'month']).size().reset_index(name='message_count')
monthly['period'] = pd.to_datetime(monthly[['year', 'month']].assign(day=1))
monthly = monthly.sort_values('period')
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly['period'], monthly['message_count'], color='#4C72B0', linewidth=2)
ax.set_title('Message volume over time')
plt.tight_layout()
plt.show()

# VIF
vif_data = df_transformed.dropna()
vif_df = pd.DataFrame({
    'feature': vif_data.columns,
    'VIF': [variance_inflation_factor(vif_data.values, i) for i in range(vif_data.shape[1])]
})
print(vif_df)

# dummy variable trap demo
dummies = pd.get_dummies(df['user_tier'], prefix='user_tier', drop_first=True)
print(dummies.columns.tolist())

---
Patterns in the Data:
1. High correlations and VIF between text_length and word_count, may need to be combined via PCA. 
2. Days until release/Days since release also have high correlations, with less extreme VIF. 
3. Response gap in minutes between users correlates around 0.20 with message length, meaning longer messages slightly correlate with longer reply times
4. Response time maintains a median of 1 minute both between and within users across conversations. Response times within conversations may be an interesting follow-up.

---
Step 8: Data Transformation 🧱
- Reshaping (pivot, melt)
- Aggregating (groupby, agg)
- Merging (combine datasets) #N/A
- Applying Functions (apply, lambda) #N/A
- Text manipulation (str methods)

Notes:
Beyond str methods for 8e. text manipulation, I want to add supplemental columns:
- Sentiment/emotion/NLP columns to be filled later
- Separating words and messages to parse out hyperlinks, special characters, special keywords, misplaced underscores rather than spaces, slashes to separate multiple words, accommodating words that include quotations on the outside or inside (such as "happy,", "done."... other issues like "one/two", "self-important", "don't")-->  the skew value (and possibly its transform tier) will change.
These are essential for NLP analysis and appropriate word separation for sentence length, word count, etc text analysis

In [ ]:
#Reshaping - Pivot Day and Hour
pd.set_option("display.max_columns", None)
day_of_week = df['date'].dt.day_name()
hour = df['date'].dt.hour

activity_pivot = pd.pivot_table(
    pd.DataFrame({'day_of_week': day_of_week, 'hour': hour, 'message_id': df['message_id']}),
    index='day_of_week', columns='hour', values='message_id', aggfunc='count'
)

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
activity_pivot = activity_pivot.reindex(day_order)
print(activity_pivot)

In [ ]:
#Aggregating by conversation ID 
df_sorted = df.sort_values(['conversation_id', 'date'])

conversation_summary = df_sorted.groupby('conversation_id').agg(
    message_count=('message_id', 'count'),
    num_participants=('from', 'nunique'),
    start_date=('date', 'min'),
    end_date=('date', 'max'),
    avg_text_length=('text_length', 'mean'),
    avg_word_count=('word_count', 'mean'),
    avg_response_gap_between_speakers=('response_gap_mins_between_speakers', 'mean'),
).reset_index()

conversation_summary['duration_mins'] = (conversation_summary['end_date'] - conversation_summary['start_date']).dt.total_seconds() / 60
conversation_summary['was_answered'] = conversation_summary['num_participants'] > 1

# ended_by_op: who sent the LAST message in the conversation
last_message = df_sorted.groupby('conversation_id').tail(1)
ended_by_op = last_message.set_index('conversation_id')['is_op']
conversation_summary['ended_by_op'] = conversation_summary['conversation_id'].map(ended_by_op)

In [ ]:
conversation_summary.head()

In [ ]:
print(conversation_summary['was_answered'].value_counts())

In [ ]:
#Examine conversations according to counts for messages and duration in days
answered = conversation_summary[conversation_summary['was_answered']].copy()
answered['duration_days'] = answered['duration_mins'] / 1440   # 1440 minutes per day

print(answered[['message_count', 'duration_days']].describe())

In [ ]:
#create categorical bins for conversations where the original message is answered by the recipient at least once
mc_bins = [1, 2, 5, 12, 30, 100, float('inf')]
mc_labels = ['2', '3-5', '6-12', '13-30', '31-100', '100+']
answered['message_count_bucket'] = pd.cut(answered['message_count'], bins=mc_bins, labels=mc_labels)

dur_bins = [-0.001, 5/1440, 30/1440, 2/24, 1, 7, 30, 365, float('inf')]
dur_labels = ['<5min', '5-30min', '30min-2hr', '2hr-1day', '1-7days', '7-30days', '30-365days', '1yr+']
answered['duration_bucket'] = pd.cut(answered['duration_days'], bins=dur_bins, labels=dur_labels)

print(answered['message_count_bucket'].value_counts(normalize=True).sort_index())
print(answered['duration_bucket'].value_counts(normalize=True).sort_index())

---
Also building an aggregate table per user. These two tables will be joined, not merged, with the primary table based on "conversation_id" and "username" derived from the 'to' and 'from' columns, 

For the user table, we can create columns:
- the count and rate of conversations where they are OP, 
- how many messages they sent vs received overall count and average per conversation, 
- month of most messages sent (same for year), 
- average time to respond to other's messages,
- rate of messages sent based on the range they were active: eg a user active from 2005-2010 rate would be relative to those years
- avg word count, avg text length

For aggregate tables, once we perform sentiment/emotion/advanced NLP analysis, we can build profiles on users, conversations, and messages.

In [ ]:
#Aggregate Users Table
sent_agg = df.groupby('from').agg(
    messages_sent=('message_id', 'count'),
    conversations_sent_in=('conversation_id', 'nunique'),
    avg_word_count=('word_count', 'mean'),
    avg_text_length=('text_length', 'mean'),
    first_message_date=('date', 'min'),
    last_message_date=('date', 'max'),
    avg_response_time_mins=('response_gap_mins_between_speakers', 'mean'),
).reset_index().rename(columns={'from': 'user'})

op_conversations = (df[df['is_op']]
                     .groupby('from')['conversation_id']
                     .nunique()
                     .rename('op_conversations'))
sent_agg = sent_agg.merge(op_conversations, left_on='user', right_index=True, how='left')
sent_agg['op_conversations'] = sent_agg['op_conversations'].fillna(0)
sent_agg['op_rate'] = sent_agg['op_conversations'] / sent_agg['conversations_sent_in']

most_common_month = df.groupby('from')['month'].agg(lambda s: s.value_counts().idxmax()).rename('most_common_month')
most_common_year = df.groupby('from')['year'].agg(lambda s: s.value_counts().idxmax()).rename('most_common_year')
sent_agg = sent_agg.merge(most_common_month, left_on='user', right_index=True, how='left')
sent_agg = sent_agg.merge(most_common_year, left_on='user', right_index=True, how='left')

received_counts = df.groupby('to').agg(
    messages_received=('message_id', 'count'),
    conversations_received_in=('conversation_id', 'nunique'),
)
sent_agg = sent_agg.merge(received_counts, left_on='user', right_index=True, how='left')
sent_agg[['messages_received', 'conversations_received_in']] = sent_agg[['messages_received', 'conversations_received_in']].fillna(0)

sent_agg['avg_sent_per_conversation'] = sent_agg['messages_sent'] / sent_agg['conversations_sent_in']
sent_agg['avg_received_per_conversation'] = sent_agg['messages_received'] / sent_agg['conversations_received_in'].replace(0, np.nan)

sent_agg['active_range_days'] = (
    (sent_agg['last_message_date'] - sent_agg['first_message_date']).dt.total_seconds() / 86400
)
sent_agg['messages_per_active_day'] = sent_agg['messages_sent'] / sent_agg['active_range_days'].replace(0, np.nan)

user_summary = sent_agg

In [ ]:
user_summary.nlargest(5, 'avg_text_length') #Examine

---
Above, I have made a pivot table to discover hourly and day-of-week trends, beyond the monthly and yearly trends from earlier steps. In this table, you see that UTC hours 6-9 have the highest counts of message IDs, on average at around 70,000 messages taking place during this time across all days of the week. Conversely, there are steep drops between hours 16 and 17, and 17 and 18... and a sharp increase between hours 5 and 6 (each of these is +/-10,000 messages being sent between the hour marks). The differences between days of the week are less noticeable per hour, all within ranges less than 10,000. This probes interest in further analysis. The values in this table are counts, but I would like to view the averages, and discover why these trends are happening, and what's being discussed during peak or low hours. 

Then, I created aggregate tables for both conversation-level and user-level data. This will make it easy to examine trends on the conversation- and user- level and build out profiles. 

Next is Text Manipulation (my favorite part 😊):
- the main tasks are first discerning what issues exist within the text. As noted above, there are typos, hyperlinks being recognized as words, words separated by slashes being considered as a single word, other issues with characters. My thought is to subset the data by messages with any non-alphabet characters, and separate those further depending on the characters present, beginning with the most obscure and then getting more general. I'm hoping that throughout this process, I can also begin to find what terms would be fitting for the lexicon and other types of terminology or patterns to look out for.
- after ensuring the text is up to par and the lexicon is looking robust, I would like to begin with spacy, and do part of speech tagging, particularly for proper nouns from the lexicon, which will likely constitute the main topics.
- then, I would like to identify emotion-rich words in the text that could identify feelings or frequent comments in relation to those topics, and begin sentiment/emotional analysis.

In [ ]:
#Identify non-letter characters
from collections import Counter
from itertools import chain

non_alpha = df['text'].str.findall(r"[^A-Za-z ]")
char_counts = Counter(chain.from_iterable(non_alpha.dropna()))
char_freq = pd.Series(char_counts).sort_values()

In [ ]:
print(char_freq)
#print(char_freq.to_string()) #for full list

In [ ]:
import re
target_char = char_freq.index[0]
example_rows = df[df['text'].str.contains(re.escape(target_char), na=False)]
print(example_rows)

In [ ]:
df = df.drop(example_rows.index) #only a single row

In [ ]:
#Classify characters
import unicodedata

def classify_char(ch):
    if ch.isascii():
        if ch.isdigit():
            return 'digit'
        if ch.isalpha():
            return 'latin_letter'
        return 'ascii_punct_symbol'
    try:
        name = unicodedata.name(ch)
    except ValueError:
        return 'unnamed_control_or_formatting'
    cat = unicodedata.category(ch)
    if ch == '\ufffd':
        return 'encoding_artifact (replacement char)'
    if cat.startswith('M'):
        return 'combining_mark'
    if cat == 'Cf':
        return 'zero_width_format_char'
    if cat in ('Ll', 'Lu') and 'LATIN' in name:
        return 'accented_latin_letter'
    for keyword, label in [
        ('CJK', 'han_chinese_japanese_kanji'),
        ('HIRAGANA', 'japanese_hiragana'),
        ('KATAKANA', 'japanese_katakana'),
        ('HANGUL', 'korean_hangul'),
        ('CYRILLIC', 'cyrillic'),
        ('ARABIC', 'arabic'),
        ('HEBREW', 'hebrew'),
        ('GREEK', 'greek'),
        ('THAI', 'thai'),
        ('DEVANAGARI', 'devanagari'),
        ('ARMENIAN', 'armenian'),
        ('GEORGIAN', 'georgian'),
        ('ETHIOPIC', 'ethiopic'),
        ('TAMIL', 'tamil'),
        ('MALAYALAM', 'malayalam'),
        ('SINHALA', 'sinhala'),
        ('CANADIAN SYLLABICS', 'canadian_aboriginal_syllabics'),
        ('BRAILLE', 'braille'),
        ('BOX DRAWINGS', 'box_drawing'),
        ('MATHEMATICAL', 'stylized_math_alphanumeric'),
        ('FULLWIDTH', 'fullwidth_latin_variant'),
    ]:
        if keyword in name:
            return label
    if cat.startswith('S'):
        return 'symbol_or_emoji'
    return f'other:{cat}:{name[:25]}'

char_categories = char_freq.index.to_series().apply(classify_char)
category_summary = char_freq.groupby(char_categories.values).sum().sort_values(ascending=False)
print(category_summary.to_string())

In [ ]:
#Categorize symbols
EMOJI_RANGES = [
    (0x1F300, 0x1F5FF), (0x1F600, 0x1F64F), (0x1F680, 0x1F6FF),
    (0x1F900, 0x1F9FF), (0x1FA70, 0x1FAFF), (0x2600, 0x26FF),
    (0x2700, 0x27BF), (0x1F1E6, 0x1F1FF),
]
CURRENCY_SYMBOLS = set('$¢£¤¥₠€₹₽₿')
MATH_SYMBOLS = set('±×÷√∞≤≥≠∈∑∂∇')
LEGAL_TECH_SYMBOLS = set('©®™§¶')
DEGREE_SYMBOLS = set('°℃℉')

def is_emoji(ch):
    cp = ord(ch)
    return any(lo <= cp <= hi for lo, hi in EMOJI_RANGES)

def classify_char(ch):
    if ch.isascii():
        if ch.isdigit(): return 'digit'
        if ch.isalpha(): return 'latin_letter'
        return 'ascii_punct_symbol'
    try:
        name = unicodedata.name(ch)
    except ValueError:
        return 'unnamed_control_or_formatting'
    cat = unicodedata.category(ch)

    if ch == '\ufffd': return 'encoding_artifact (replacement char)'
    if is_emoji(ch): return 'pictographic_emoji'
    if ch in CURRENCY_SYMBOLS: return 'currency_symbol'
    if ch in MATH_SYMBOLS: return 'math_symbol'
    if ch in LEGAL_TECH_SYMBOLS: return 'legal_technical_symbol'
    if ch in DEGREE_SYMBOLS: return 'degree_temperature_symbol'
    if cat == 'No' and ('FRACTION' in name or 'SUPERSCRIPT' in name or 'SUBSCRIPT' in name):
        return 'number_form (fraction/exponent)'
    if cat.startswith('M'): return 'combining_mark'
    if cat == 'Cf': return 'zero_width_format_char'
    if cat in ('Ll', 'Lu') and 'LATIN' in name: return 'accented_latin_letter'
    if 'IDEOGRAPHIC' in name or 'FULLWIDTH' in name: return 'cjk_typographic_punctuation'
    if cat in ('Pi', 'Pf', 'Pd', 'Ps', 'Pe', 'Pc', 'Po'):
        return 'typographic_punctuation (quotes/dashes/etc)'
    if cat == 'Zs': return 'unusual_whitespace'
    for keyword, label in [
        ('CJK', 'han_chinese_japanese_kanji'), ('HIRAGANA', 'japanese_hiragana'),
        ('KATAKANA', 'japanese_katakana'), ('HANGUL', 'korean_hangul'),
        ('CYRILLIC', 'cyrillic'), ('ARABIC', 'arabic'), ('HEBREW', 'hebrew'),
        ('GREEK', 'greek'), ('THAI', 'thai'), ('DEVANAGARI', 'devanagari'),
        ('ARMENIAN', 'armenian'), ('GEORGIAN', 'georgian'), ('ETHIOPIC', 'ethiopic'),
        ('TAMIL', 'tamil'), ('MALAYALAM', 'malayalam'), ('SINHALA', 'sinhala'),
        ('BRAILLE', 'braille'), ('BOX DRAWINGS', 'box_drawing'),
        ('MATHEMATICAL', 'stylized_math_alphanumeric'),
    ]:
        if keyword in name: return label
    if cat.startswith('S'): return 'other_symbol'
    return f'other:{cat}:{name[:25]}'

char_categories = char_freq.index.to_series().apply(classify_char)
category_summary = char_freq.groupby(char_categories.values).sum().sort_values(ascending=False)
print(category_summary)
#print(category_summary.to_string()) #paste this in a below cell for full list

In [ ]:
#Replace characters with frequency less than 10 in the entire dataset with a flower emoji - temporary for text_cleaned
rare_chars = char_freq[char_freq < 10].index.tolist()
pattern = '[' + ''.join(re.escape(c) for c in rare_chars) + ']'
df['text_cleaned'] = df['text'].str.replace(pattern, '🪻', regex=True)

In [ ]:
df[df['text_cleaned'].str.contains('🪻', na=False)][['message_id', 'conversation_id', 'date', 'from', 'to', 'text','text_cleaned']] #examine

---
Im thinking for the cleaned text column... if theres entire words/characters in another language, they should be replaced with their character category like (CYRILLIC). not per character, but until a space or punctuation symbol interrupts it. 

As I review these, I'm discovering sometimes characters like :Д and ヅ are flagging as languages, even though they are used in these cases as emoticons.
Fixing language-based issues like this as I review.

In [ ]:
#Spot replacing emojis that feature characters from other languages
def qualifying_conversations(df, char_categories, label, target_char):
    label_chars = sorted(char_categories[char_categories == label].index.tolist())
    flags = pd.DataFrame(index=df.index)
    for ch in label_chars:
        flags[ch] = df['text'].str.contains(re.escape(ch), na=False, regex=True)
    conv_flags = flags.groupby(df['conversation_id']).any()
    num_distinct = conv_flags.sum(axis=1)
    qualifying = conv_flags[(num_distinct == 1) & (conv_flags.get(target_char, False))].index
    return set(qualifying)

cyr_qualifying = qualifying_conversations(df, char_categories, 'cyrillic', 'Д')
kata_qualifying = qualifying_conversations(df, char_categories, 'japanese_katakana', 'ヅ')

df['text_cleaned'] = df['text']

cyr_mask = df['conversation_id'].isin(cyr_qualifying)
df.loc[cyr_mask, 'text_cleaned'] = df.loc[cyr_mask, 'text_cleaned'].str.replace(
    r'(^|\s):Д(\s|$)', r'\1:)\2', regex=True
)

kata_mask = df['conversation_id'].isin(kata_qualifying)
df.loc[kata_mask, 'text_cleaned'] = df.loc[kata_mask, 'text_cleaned'].str.replace(
    r'(^|\s)ヅ(\s|$)', r'\1:)\2', regex=True
)

In [ ]:
#Language labels
NON_LATIN_LABELS = ['han_chinese_japanese_kanji','japanese_hiragana','japanese_katakana',
                     'korean_hangul','cyrillic','arabic','hebrew','greek','thai','devanagari',
                     'armenian','georgian','ethiopic','malayalam','tamil','sinhala']

for label in NON_LATIN_LABELS:
    chars = char_categories[char_categories == label].index.tolist()
    if not chars:
        continue
    pattern = '[' + ''.join(re.escape(c) for c in chars) + ']+'
    token = label.upper()
    df['text_cleaned'] = df['text_cleaned'].str.replace(pattern, token, regex=True)

In [ ]:
#For non-latin language labels many times in sequence, compress to one single label:
for label in NON_LATIN_LABELS:
    token = label.upper()
    pattern = rf'\b{re.escape(token)}(?:\s+{re.escape(token)})+\b'
    df['text_cleaned'] = df['text_cleaned'].str.replace(pattern, token, regex=True)

In [ ]:
label_pattern = '|'.join(label.upper() for label in NON_LATIN_LABELS)

In [ ]:
df[df['text_cleaned'].str.contains(label_pattern, na=False, regex=True)][['message_id', 'conversation_id', 'date', 'from', 'to', 'text', 'text_cleaned']].sample(n=5) #examine

In [ ]:
df[['message_id', 'conversation_id', 'date', 'from', 'to', 'text', 'text_cleaned']].sample(n=10) #examine

I refreshed the above sample many times to begin developing the Ubuntu jargon lexicon, then provided it to AI to round out prevalent developer terms that are relevant to this era of the internet. Further, I built a lexicon for slang that was common at the time, as well as emojis, kaomojis, emoticons, and stylistic characters that imbue meaning and can and should influence message sentiment. 

The next step is applying these symbolic interpretations to text detection before interpretation. Messages containing the placeholder emoji, 🪻, which was released in 2022 (after the most recent message in the dataset and therefore entirely unique), will be re-examined within context of both the message itself and its language, and the slang lexicon. 

Another development was creating templates for emails, website domains, phone numbers, SSNs units of measurement, and IP addresses, so that these can be parsed and recognized by the program. This was worthwhile for this dataset and as a template for future projects. 

After accounting for these topics, the amount of undiscernable text will ideally be vastly reduced. I will examine the remaining non-english-word text, to determine if they are latin/roman alphabet words in other languages, or other obscure topics that can be separated. 

The rest can ideally be distinguished as typos - effectively rendering the text of each message as cleaned and interpretable for the program and for human readers alike.

---

In [ ]:
# Apply structural anonymization, lexicons, slang, and glued-term matching.
# apply_lexicons delegates row-parallel work to pipeline.parallel_execution.
from pipeline.apply_lexicons import apply_lexicons

workers = workers_for(len(df))
df, lexicon_matches_df, lexicon_tally = apply_lexicons(
    df,
    text_col='text_cleaned',
    workers=workers,
)
no_match_df = df.loc[~df['has_lexicon_match']].copy()
print(f'{len(lexicon_matches_df):,} matched rows; {len(no_match_df):,} rows remain unmatched')


In [ ]:
#Counts per category and explicit mentions in cleaned message text
print(df[['email_count','domain_count','phone_count','ipv4_count',
           'ipv6_count','menu_path_count','keyboard_shortcut_count']].sum())
print(df['text_cleaned'].str.contains(
    'EMAILADDRESS|WEBSITEDOMAIN|PHONENUMBER|IPADDRESS|IPV6ADDRESS|MENUPATH|KEYBOARDSHORTCUT',
    na=False, regex=True
).sum())

In [ ]:
lexicon_tally['tech'].most_common(20)

In [ ]:
lexicon_tally['slang'].most_common(20)

In [ ]:
lexicon_tally['anonymized']

---
I attempted to run apply lexicons with unilateral CPU processing, and waited over 1.5 hours before disrupting the kernel. After this, I decided to rewrite the process using parallel processing across my laptop's 8 cores, which cut the time down to about 40 minutes for the entire task

---

In [ ]:
lexicon_matches_df.sample(5)[['message_id','conversation_id','date','from','to','text','text_cleaned']]

In [ ]:
lexicon_matches_df.shape

In [ ]:
df.shape

---
the lexicon_matches_df is over HALF the size of the entire dataframe! This is big news, half of the messages can be accounted for. Now, we will examine the rest of the dataframe that does not have matches, to see what is going on, and how we may be able to group the messages further

In [ ]:
#Create a dataframe for messages with no matches
no_match_df = df[~df['has_lexicon_match']].copy()
print(f"{len(df):,} total rows")
print(f"{len(lexicon_matches_df):,} matched ({len(lexicon_matches_df)/len(df):.1%})")
print(f"{len(no_match_df):,} unmatched ({len(no_match_df)/len(df):.1%})")

---
Within no_match_df, while examining, I discovered there are "glued_matches" which contain real words that are combined due to improper spacing or other characters distorting them. It is essential that we address these later. The canonical detector is `pipeline/glued_terms.py`.

In [ ]:
#Discover these glued_matches
import time
import pandas as pd
from tqdm.auto import tqdm
from pipeline.glued_terms import find_glued_matches

texts = no_match_df['text_cleaned'].astype(str)
workers = workers_for(len(texts))
print(f"[{time.strftime('%H:%M:%S')}] starting on {len(texts):,} rows with {workers} worker(s)")
t0 = time.time()
results = map_rows(find_glued_matches, texts, workers=workers)
no_match_df['glued_matches'] = pd.Series(results, index=texts.index, dtype=object)
print(f"[{time.strftime('%H:%M:%S')}] done ({time.time()-t0:.1f}s)")
no_match_df['glued_match_count'] = no_match_df['glued_matches'].str.len()

In [ ]:
#Examine word count
no_match_df['word_count'] = no_match_df['text_cleaned'].str.split().str.len()
matched_word_count = lexicon_matches_df['text_cleaned'].str.split().str.len()

print("matched   :", matched_word_count.describe())
print("unmatched :", no_match_df['word_count'].describe())

In [ ]:
#Examine common no match words
from collections import Counter
import re

sample = no_match_df['text_cleaned'].sample(min(300_000, len(no_match_df)), random_state=0)
word_re = re.compile(r"[a-zA-Z']+")
counts = Counter()
for text in sample:
    counts.update(w.lower() for w in word_re.findall(text))

print(counts.most_common(50)) # dominant vocabulary
rare = [w for w, c in counts.items() if c <= 3]
print(len(rare), "rare/singleton terms out of", len(counts))

In [ ]:
print([item for item in counts.most_common() if item[1] > 100][:-200:-5]) #Look at words within certain frequency ranks

In [ ]:
non_ascii_mask = no_match_df['text_cleaned'].str.contains(r'[^\x00-\x7F]', regex=True, na=False)
print(f"{non_ascii_mask.sum():,} unmatched rows have non-ASCII chars ({non_ascii_mask.mean():.1%})")
no_match_df.loc[non_ascii_mask, 'text_cleaned'].sample(20, random_state=1).tolist()

---
Interim: Check heavy memory usage

---

In [ ]:
df.memory_usage(deep=True).sort_values(ascending=False) / 1e6   # MB per column

In [ ]:
import psutil, os
rss_gb = psutil.Process(os.getpid()).memory_info().rss / 1e9
print(f"process RSS: {rss_gb:.2f} GB")

In [ ]:
def size_of(obj):
    if isinstance(obj, pd.DataFrame):
        return obj.memory_usage(deep=True).sum()
    if isinstance(obj, pd.Series):
        return obj.memory_usage(deep=True)
    return sys.getsizeof(obj)   # rough floor for everything else -- lists of objects will read low

ns = get_ipython().user_ns   # notebook namespace, includes Out/In history too
rows = []
for name, obj in list(ns.items()):
    if name.startswith('_'):
        continue
    try:
        rows.append((name, type(obj).__name__, size_of(obj) / 1e6))
    except Exception:
        pass

pd.DataFrame(rows, columns=['name', 'type', 'MB']).sort_values('MB', ascending=False).head(5)

In [ ]:
#Delete Unused Vars
del char_freq, non_alpha, example_rows, target_char, sample, counts, non_ascii_mask, matched_word_count
del df_sorted, day_of_week, hour, outlier_flags, prev_from, speaker_changed, df_transformed, char_categories

In [ ]:
#For conversation- and user- aggregated tables from earlier, replace avg_word_count and avg_text_length cols with NA, since they need to be re-made later
for table in (conversation_summary, user_summary):
    table[['avg_word_count', 'avg_text_length']] = table[['avg_word_count', 'avg_text_length']].apply(
        lambda col: pd.to_numeric(pd.Series(1, index=col.index), downcast='unsigned'))

---
Now, we will filter through all the cleaned, non-matched dataframe words, see which ones are known english words. from there, we can deduce which are not even english but still in latin/roman characters, are more missed jargon, are usernames, or are typos.

---

In [ ]:
# Extract residual vocabulary with the canonical pipeline stage.
# extract_residual_vocabulary also delegates parallel work to parallel_execution.
from pipeline.residual_stage import extract_residual_vocabulary

workers = workers_for(len(df))
df, residual_counts = extract_residual_vocabulary(
    df,
    text_col='text_cleaned',
    workers=workers,
)
no_match_df = df.loc[df['has_residual']].copy()
unresolved_words = set(residual_counts)

residual_review = pd.DataFrame(
    residual_counts.most_common(),
    columns=['word', 'total_count'],
)
output_path = OUTPUT_DIR / 'residual_words_for_classification.csv'
residual_review.to_csv(output_path, index=False)
print(f'{len(residual_review):,} residual words exported to {output_path}')
residual_review.head(20)


---
Now that we have filtered through the dataframe, using python's word checker, we have produced several outputs of value. Firstly, we have identified messages comprised solely of real, non-lexicon words, and: 
1. added these rows to lexicon-matches dataframe
2. removed these rows from the no-matches dataframe
2. established a binary column in the main dataframe (all_words_english_known) where these are found "True"

Two outstanding areas to resolve and improve upon now:

1. Firstly, performing human-in-the-loop review on some of the outputs produced in the above cell. Namely: the latin-character containing non-English "words", the plain non-English "words" (used loosely), and the "words" containing digits. Since the cell just above ran a test twice to detect whether words were English; the first time stripping the message text of non-alphabetical characters, which were allowed to remain the second time; the above categories (latin/plain/numeric) of "words" were compiled into mutually exclusive lists, depending on whether they passed neither test, or only failed the test that did not strip the non-alphabetic characters.

- After reviewing these lists, I will add the hand-selected words (particularly common typos) to be included in the lexicons, which will also help with future renditions of rule-based NLP analysis, while remaining relatively computationally inexpensive. These typos are valuable as they occur sometimes thousands of times, meaning that they very likely extrapolate to other forum-based chat data.

2. Secondly, resolving the outstanding glued-matches issue (which were words that matched with the lexicon when parsed but not in their current format), and applying these changes across all dataframes. 

The remaining "words" that passed one or neither of the above cell's tests once ran again, as well as the unresolved glued matches, will be fed into claude AI for one reason: I am certain much of it is Linux/Developer specific jargon and codewords that could map to certain hot-topics, but I lack the subject matter expertise to identify very specific lingo. Ai will sort all these remaining "words" into either a jargon bucket, replaced with language labels (upon detecting real-non english words with roman characters, e.g., ITALIAN, SPANISH), and the rest will be replaced simply with the keyword "NONWORD". Standalone numbers will be preserved, rare typos corrected. 

Then, all the dataframe where all_words_english_known = FALSE will be re-ran with these impending changes and updates, and the supplemental dataframes will be wiped from memory, as they are no longer necessary.

---

In [ ]:
# Inspect the highest-frequency residual vocabulary.
residual_review.head(50)


Typos discovered from these 6 series were added to `reviewed_overlays/chat_normalization.py`.

Next we will run it and update dataframes accordingly, then address any remaining glued matches that were not resolved and fix them unilaterally.

In [ ]:
#apply human-validated chat normalization overlay across all dataframes
from reviewed_overlays.chat_normalization import apply_latin_corrections, normalize_chat_shorthand

for name, frame in [('df', df), ('no_match_df', no_match_df), ('lexicon_matches_df', lexicon_matches_df)]:
    frame['text_cleaned'] = normalize_chat_shorthand(apply_latin_corrections(frame['text_cleaned']))
    print(f"{name}: chat-normalization applied to {len(frame):,} rows")

In [ ]:
no_match_df['text'].str.contains(' dont ').sum()

In [ ]:
# Summarize glued-term candidates already produced by apply_lexicons.
import re
from collections import defaultdict

tally = defaultdict(lambda: {'count': 0, 'examples': set()})
for matches in df['glued_matches']:
    for match in matches:
        key = (match['glue'], match['term'])
        tally[key]['count'] += 1
        if len(tally[key]['examples']) < 5:
            tally[key]['examples'].add(match['run'])

glued_review = pd.DataFrame([
    {
        'glue': glue,
        'term': term,
        'count': values['count'],
        'examples': sorted(values['examples']),
    }
    for (glue, term), values in tally.items()
])

if not glued_review.empty:
    glue_risk = {'prefix': 0, 'suffix': 1, 'infix': 2}
    glued_review['glue_risk'] = glued_review['glue'].map(glue_risk)
    glued_review = (
        glued_review
        .sort_values(['glue_risk', 'count'], ascending=[True, False])
        .drop(columns='glue_risk')
    )

    def is_digit_suffixed(row):
        if row['glue'] != 'prefix':
            return False
        return all(
            re.fullmatch(rf'{re.escape(row["term"])}\d+', example, re.IGNORECASE)
            for example in row['examples']
        )

    glued_review['safe_to_batch_approve'] = glued_review.apply(is_digit_suffixed, axis=1)
else:
    glued_review['safe_to_batch_approve'] = pd.Series(dtype=bool)

output_path = OUTPUT_DIR / 'glued_match_review.csv'
glued_review.to_csv(output_path, index=False)
print(f'{len(glued_review):,} glued-term candidates exported to {output_path}')
glued_review.head(40)


In [ ]:
# Extract residual vocabulary with the canonical pipeline stage.
# extract_residual_vocabulary also delegates parallel work to parallel_execution.
from pipeline.residual_stage import extract_residual_vocabulary

workers = workers_for(len(df))
df, residual_counts = extract_residual_vocabulary(
    df,
    text_col='text_cleaned',
    workers=workers,
)
no_match_df = df.loc[df['has_residual']].copy()
unresolved_words = set(residual_counts)

residual_review = pd.DataFrame(
    residual_counts.most_common(),
    columns=['word', 'total_count'],
)
output_path = OUTPUT_DIR / 'residual_words_for_classification.csv'
residual_review.to_csv(output_path, index=False)
print(f'{len(residual_review):,} residual words exported to {output_path}')
residual_review.head(20)


In [ ]:
# Apply structural anonymization, lexicons, slang, and glued-term matching.
# apply_lexicons delegates row-parallel work to pipeline.parallel_execution.
from pipeline.apply_lexicons import apply_lexicons

workers = workers_for(len(df))
df, lexicon_matches_df, lexicon_tally = apply_lexicons(
    df,
    text_col='text_cleaned',
    workers=workers,
)
no_match_df = df.loc[~df['has_lexicon_match']].copy()
print(f'{len(lexicon_matches_df):,} matched rows; {len(no_match_df):,} rows remain unmatched')


---
At this point, remaining vocabulary can be sent through the configured human-review or API-classification path.

---

In [ ]:
#Delete the other dataframes
import gc

# Delete the DataFrame variable
del no_match_df
del lexicon_matches_df

In [ ]:
# One-time repair of the doesn't't / didn't't 
hits = df['text_cleaned'].str.contains(r"doesn't't|didn't't", regex=True, na=False).sum()
df['text_cleaned'] = (
    df['text_cleaned']
    .str.replace(r"\bdoesn't't\b", "doesn't", regex=True)
    .str.replace(r"\bdidn't't\b", "didn't", regex=True)
)
print(f"repaired {hits:,} rows")

In [ ]:
#Re-run human-validated chat normalization overlay
from reviewed_overlays.chat_normalization import apply_latin_corrections, normalize_chat_shorthand

df['text_cleaned'] = normalize_chat_shorthand(apply_latin_corrections(df['text_cleaned']))
print(f"chat-normalization re-applied to {len(df):,} rows")

In [ ]:
# Apply the committed human-reviewed residual overlay.
# The canonical stage loads reviewed_overlays internally and uses the shared
# parallel execution backend for residual extraction.
from pipeline.residual_stage import classify_residuals, extract_residual_vocabulary

workers = workers_for(len(df))
df, residual_counts = extract_residual_vocabulary(
    df,
    text_col='text_cleaned',
    workers=workers,
)
df, residual_labels, residual_sources = classify_residuals(
    df,
    residual_counts,
    policy='reviewed',
    text_col='text_cleaned',
)
print(f'Applied {len(residual_labels):,} reviewed residual labels to {len(df):,} rows')


In [ ]:
df.head()

---
Step 8.5: Sentiment Analysis and Advanced NLP

Text manipulation is complete. This stage scores `text_cleaned` with VADER, the transformer, or both, then writes the single final sentiment-enriched artifact. Transformer dtype is explicit so a GPU never silently selects fp16.

---

In [ ]:
# Sentiment stage. Choose one: 'vader', 'transformer', or 'both'.
sentiment_mode = 'vader'
transformer_batch_size = 32
transformer_device = None  # None selects an available device.
transformer_dtype = 'float32'  # Explicit; never silently switch to fp16.

from pipeline.sentiment_analysis import analyze_sentiment

valid_modes = {'vader', 'transformer', 'both'}
if sentiment_mode not in valid_modes:
    raise ValueError(f'sentiment_mode must be one of {sorted(valid_modes)}')

# Repeated runs replace, rather than mix, columns from an earlier mode.
sentiment_columns = [
    'vader_compound', 'vader_label',
    'transformer_label', 'transformer_score',
]
df = df.drop(columns=[column for column in sentiment_columns if column in df.columns])

df = analyze_sentiment(
    df,
    text_column='text_cleaned',
    mode=sentiment_mode,
    transformer_batch_size=transformer_batch_size,
    transformer_device=transformer_device,
    transformer_dtype=transformer_dtype,
)

print(f'Applied {sentiment_mode} sentiment analysis to {len(df):,} rows')
print('Sentiment columns:', [column for column in sentiment_columns if column in df.columns])
print('The Step 8 artifact is saved only after advanced-NLP validation below.')


---
Step 8.6: Advanced NLP

This final Step 8 stage adds compact spaCy token/POS summaries and named
entities, then fits one deterministic NMF topic model on a bounded sample and
uses that same model for every row. Ubuntu technical entities remain owned by
the earlier canonical lexicon columns (`tech_lexicon_matches` and
`tech_lexicon_match_count`) instead of being counted twice.

Transformer emotion classification is optional and defaults off. When enabled,
its device and dtype are explicit; GPU detection never silently selects fp16.
Full token-level JSON is also opt-in because it is not a practical default for
8.6 million messages. A normalized token table belongs in the later
Databricks implementation.

---


In [ ]:
# Configure and run the remaining Step 8 NLP stages.
from pipeline.advanced_nlp import annotate_messages

run_spacy = True
include_token_details = False  # Keep the full-corpus output compact.
topic_mode = 'nmf'
topic_count = 12
topic_fit_sample_size = 100_000

# Optional transformer arm. Leave disabled until the controlled GPU dtype
# comparison is complete; when enabled, float32 remains explicit.
emotion_mode = 'none'  # Change to 'transformer' to opt in.
emotion_device = None
emotion_dtype = 'float32'
emotion_batch_size = 32

advanced_workers = workers_for(len(df))
df = annotate_messages(
    df,
    text_col='text_cleaned',
    run_spacy=run_spacy,
    workers=advanced_workers,
    include_token_details=include_token_details,
    topic_mode=topic_mode,
    n_topics=topic_count,
    topic_fit_sample_size=topic_fit_sample_size,
    emotion_mode=emotion_mode,
    emotion_device=emotion_device,
    emotion_dtype=emotion_dtype,
    emotion_batch_size=emotion_batch_size,
)

print(f'Advanced NLP complete with {advanced_workers} local worker(s)')
print('Topic model fit rows:', df.attrs['topic_model_bundle'].fit_rows)


In [ ]:
# Yelp-style fail-closed validation: do not save malformed or frozen features.
from pipeline.advanced_nlp import validate_advanced_nlp

advanced_validation = validate_advanced_nlp(
    df,
    text_col='text_cleaned',
    run_spacy=run_spacy,
    topic_mode=topic_mode,
    emotion_mode=emotion_mode,
    require_technical_lexicon=True,
)
print('Advanced NLP validation:', advanced_validation)

# The sole notebook pickle export now occurs after every Step 8 feature passes.
output_path = OUTPUT_DIR / 'df_with_sentiment.pkl'
df.to_pickle(output_path)
print(f'Saved {len(df):,} rows, {len(df.columns)} columns -> {output_path}')


---
Step 8 complete

The message-level dataset now contains the completed cleaning and review
overlays, sentiment, compact spaCy/POS and named-entity features, stable topic
assignments, and optional transformer emotion fields when enabled.

Step 9 begins the silver-to-gold layer: conversation-, user-, channel-, and
time-level aggregation using these validated message features.

---


---
Step 9: Statistics & Correlation

Now that the message-level features are validated, we can build bounded
conversation-level analysis tables. The sample is selected by complete
conversation, not by isolated message, so response outcomes and temporal
features remain internally consistent.

The statistical tests below are exploratory rather than causal. Because the
full corpus is extremely large, effect sizes and confidence intervals matter
more than p-values alone.

---


In [ ]:
# Build a deterministic, conversation-complete analysis sample.
from pipeline.aggregation import run_silver_to_gold

ANALYSIS_SEED = 42
ANALYSIS_CONVERSATION_LIMIT = 100_000
PLOT_MESSAGE_LIMIT = 50_000

conversation_ids = pd.Index(df['conversation_id'].dropna().unique())
analysis_rng = np.random.default_rng(ANALYSIS_SEED)

if len(conversation_ids) > ANALYSIS_CONVERSATION_LIMIT:
    selected_conversation_ids = analysis_rng.choice(
        conversation_ids.to_numpy(),
        size=ANALYSIS_CONVERSATION_LIMIT,
        replace=False,
    )
    analysis_scope = 'deterministic conversation sample'
else:
    selected_conversation_ids = conversation_ids.to_numpy()
    analysis_scope = 'all conversations'

analysis_messages = df[df['conversation_id'].isin(selected_conversation_ids)].copy()
analysis_messages['_analysis_date'] = pd.to_datetime(
    analysis_messages['date'], errors='coerce', utc=True
)
sort_columns = ['conversation_id', '_analysis_date']
if 'message_id' in analysis_messages.columns:
    sort_columns.append('message_id')
analysis_messages = analysis_messages.sort_values(sort_columns, kind='stable')

# One initial message per conversation prevents later messages from leaking
# into the Step 11 response model.
first_messages = analysis_messages.groupby(
    'conversation_id', sort=False, observed=True, dropna=False
).head(1).copy()
sender_counts = analysis_messages.groupby(
    'conversation_id', observed=True, dropna=False
)['from'].nunique()
first_messages['was_answered'] = first_messages['conversation_id'].map(
    sender_counts.gt(1)
).fillna(False).astype(bool)

# Gold tables use the same validated aggregation functions as the local CLI
# and the Spark/Delta implementation.
conversation_gold = run_silver_to_gold(
    analysis_messages.drop(columns=['_analysis_date']),
    gold_level='conversation',
    engineer=False,
)
monthly_gold = run_silver_to_gold(
    analysis_messages.drop(columns=['_analysis_date']),
    gold_level='date',
    date_granularity='month',
    engineer=False,
)

release_gold = None
if 'days_since_release_bucket' in analysis_messages.columns:
    release_gold = run_silver_to_gold(
        analysis_messages.drop(columns=['_analysis_date']),
        gold_level='release',
        release_axis='since',
        engineer=False,
    )

print(f'Analysis scope: {analysis_scope}')
print(f'{len(analysis_messages):,} messages across {len(first_messages):,} conversations')
print(f'Observed answer rate: {first_messages["was_answered"].mean():.1%}')


In [ ]:
# Statistical summaries and a robust Spearman correlation matrix.
numeric_candidates = [
    'text_length', 'word_count', 'is_op', 'tech_lexicon_match_count',
    'slang_match_count', 'glued_match_count', 'vader_compound',
    'transformer_expected_sentiment', 'transformer_score',
    'transformer_normalized_entropy', 'nlp_lexical_diversity',
    'nlp_sentence_count', 'nlp_avg_sentence_tokens',
    'nlp_negation_count', 'nlp_punctuation_count', 'noun_ratio',
    'proper_noun_ratio', 'verb_ratio', 'adjective_ratio', 'adverb_ratio',
    'named_entity_count', 'topic_score', 'topic_entropy',
]
analysis_numeric = [
    column for column in numeric_candidates
    if column in first_messages.columns
    and pd.api.types.is_numeric_dtype(first_messages[column])
    and first_messages[column].notna().any()
]

correlation_frame = first_messages[analysis_numeric].copy()
correlation_frame['was_answered'] = first_messages['was_answered'].astype('int8')
correlation_matrix = correlation_frame.corr(method='spearman', min_periods=100)
statistical_summary = correlation_frame.describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

display(statistical_summary.round(3))
display(
    correlation_matrix['was_answered']
    .drop('was_answered')
    .sort_values(key=lambda values: values.abs(), ascending=False)
    .rename('spearman_with_answered')
    .to_frame()
    .round(3)
)


In [ ]:
# Hypothesis tests with effect sizes and Holm correction for multiple tests.
from scipy import stats

hypothesis_rows = []

def add_mann_whitney_test(name, feature, group_mask, group_1, group_0):
    values = pd.to_numeric(first_messages[feature], errors='coerce')
    group_mask = pd.Series(group_mask, index=first_messages.index, dtype='boolean')
    eligible = group_mask.notna()
    sample_1 = values[eligible & group_mask.fillna(False)].dropna()
    sample_0 = values[eligible & ~group_mask.fillna(False)].dropna()
    if len(sample_1) < 30 or len(sample_0) < 30:
        return
    statistic, p_value = stats.mannwhitneyu(
        sample_1, sample_0, alternative='two-sided', method='asymptotic'
    )
    rank_biserial = (2 * statistic / (len(sample_1) * len(sample_0))) - 1
    hypothesis_rows.append({
        'hypothesis': name,
        'test': 'Mann-Whitney U',
        'feature': feature,
        'group_1': group_1,
        'group_0': group_0,
        'n_1': len(sample_1),
        'n_0': len(sample_0),
        'median_1': sample_1.median(),
        'median_0': sample_0.median(),
        'effect_size': rank_biserial,
        'p_value': p_value,
    })

sentiment_feature = next((
    column for column in ('transformer_expected_sentiment', 'vader_compound')
    if column in first_messages.columns
), None)

if sentiment_feature:
    add_mann_whitney_test(
        'Initial-message sentiment differs by response outcome',
        sentiment_feature,
        first_messages['was_answered'],
        'answered',
        'unanswered',
    )
    if 'is_op' in first_messages.columns:
        add_mann_whitney_test(
            'Initial-message sentiment differs by OP status',
            sentiment_feature,
            first_messages['is_op'].fillna(False).astype(bool),
            'original poster',
            'other user',
        )

if 'days_since_release' in first_messages.columns:
    release_days = pd.to_numeric(
        first_messages['days_since_release'], errors='coerce'
    )
    near_release = release_days.le(30).where(release_days.notna())
    add_mann_whitney_test(
        'Initial-message length differs near an Ubuntu release',
        'word_count',
        near_release,
        '0-30 days since release',
        'more than 30 days since release',
    )

# Topic and response outcome are both categorical, so use chi-square and
# Cramer's V rather than treating topic IDs as continuous numbers.
if 'topic_label' in first_messages.columns:
    topic_table = pd.crosstab(
        first_messages['topic_label'], first_messages['was_answered']
    )
    topic_table = topic_table.loc[topic_table.sum(axis=1) >= 30]
    if topic_table.shape[0] > 1 and topic_table.shape[1] == 2:
        chi2, p_value, degrees_of_freedom, expected = stats.chi2_contingency(topic_table)
        denominator = topic_table.to_numpy().sum() * min(topic_table.shape[0] - 1, 1)
        cramers_v = np.sqrt(chi2 / denominator) if denominator else np.nan
        hypothesis_rows.append({
            'hypothesis': 'Topic assignment and response outcome are associated',
            'test': 'Chi-square',
            'feature': 'topic_label',
            'group_1': 'topic',
            'group_0': 'was_answered',
            'n_1': int(topic_table.to_numpy().sum()),
            'n_0': int(topic_table.shape[0]),
            'median_1': np.nan,
            'median_0': np.nan,
            'effect_size': cramers_v,
            'p_value': p_value,
        })

hypothesis_columns = [
    'hypothesis', 'test', 'feature', 'group_1', 'group_0',
    'n_1', 'n_0', 'median_1', 'median_0', 'effect_size', 'p_value',
]
hypothesis_tests = pd.DataFrame(hypothesis_rows, columns=hypothesis_columns)
if not hypothesis_tests.empty:
    order = np.argsort(hypothesis_tests['p_value'].to_numpy())
    sorted_p = hypothesis_tests['p_value'].to_numpy()[order]
    holm = np.maximum.accumulate(
        sorted_p * (len(sorted_p) - np.arange(len(sorted_p)))
    ).clip(0, 1)
    adjusted = np.empty_like(holm)
    adjusted[order] = holm
    hypothesis_tests['p_value_holm'] = adjusted
    hypothesis_tests['significant_at_0_05'] = hypothesis_tests['p_value_holm'] < 0.05

display(hypothesis_tests.round(4))


---
Step 9 interpretation notes

- Spearman correlation is used because message lengths, activity, and response
  times are heavily skewed.
- Mann-Whitney tests do not assume normally distributed sentiment or length.
- Rank-biserial correlation and Cramer's V report practical effect size.
- Holm-adjusted p-values limit false positives across the exploratory tests.
- Statistical association does not establish that wording, topic, or release
  timing causes a response.

---


---
Step 10: Visualization

These figures follow the Yelp project's descriptive and diagnostic views, but
use Ubuntu-specific outcomes: sentiment, response likelihood, release cycle,
topic, and message volume. Plotting uses bounded samples or aggregated tables
so the notebook never sends millions of marks to Matplotlib.

Every figure is saved at 300 DPI under outputs/figures.

---


In [ ]:
# Publication-ready plot defaults and deterministic plotting sample.
import matplotlib.pyplot as plt
import seaborn as sns

FIGURE_DIR = OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

UBUNTU_ORANGE = '#E95420'
AUBERGINE = '#772953'
WARM_GRAY = '#AEA79F'
DARK_INK = '#2C2C2C'

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'axes.titleweight': 'bold',
    'axes.labelcolor': DARK_INK,
    'text.color': DARK_INK,
    'figure.facecolor': 'white',
})

plot_messages = analysis_messages.drop(columns=['_analysis_date']).copy()
if len(plot_messages) > PLOT_MESSAGE_LIMIT:
    plot_messages = plot_messages.sample(
        PLOT_MESSAGE_LIMIT, random_state=ANALYSIS_SEED
    ).sort_index(kind='stable')

saved_figures = []

def save_figure(fig, filename):
    path = FIGURE_DIR / filename
    fig.savefig(path, bbox_inches='tight', facecolor='white')
    saved_figures.append(path)
    return path

print(f'Plotting {len(plot_messages):,} bounded message rows')


In [ ]:
# Distribution and relationship plots.
score_columns = [
    column for column in ('vader_compound', 'transformer_expected_sentiment')
    if column in plot_messages.columns
]
if score_columns:
    score_long = plot_messages[score_columns].melt(
        var_name='sentiment_method', value_name='sentiment_score'
    ).dropna()
    fig, ax = plt.subplots(figsize=(10, 5.5))
    sns.histplot(
        data=score_long,
        x='sentiment_score',
        hue='sentiment_method',
        bins=50,
        stat='density',
        common_norm=False,
        element='step',
        palette=[UBUNTU_ORANGE, AUBERGINE][:len(score_columns)],
        ax=ax,
    )
    ax.set(title='Sentiment score distributions', xlabel='Sentiment score', ylabel='Density')
    save_figure(fig, 'sentiment_distributions.png')
    plt.show()

if sentiment_feature:
    relationship_plot = first_messages[[sentiment_feature, 'was_answered']].dropna().copy()
    relationship_plot['response_outcome'] = relationship_plot['was_answered'].map({
        False: 'Unanswered', True: 'Answered'
    })
    fig, ax = plt.subplots(figsize=(8, 5.5))
    sns.violinplot(
        data=relationship_plot,
        x='response_outcome',
        y=sentiment_feature,
        hue='response_outcome',
        order=['Unanswered', 'Answered'],
        hue_order=['Unanswered', 'Answered'],
        palette=[WARM_GRAY, UBUNTU_ORANGE],
        legend=False,
        inner='quartile',
        cut=0,
        ax=ax,
    )
    ax.set(title='Initial-message sentiment by response outcome', xlabel='', ylabel='Sentiment')
    save_figure(fig, 'sentiment_by_response_outcome.png')
    plt.show()

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    correlation_matrix,
    cmap=sns.diverging_palette(280, 25, as_cmap=True),
    center=0,
    vmin=-1,
    vmax=1,
    square=False,
    linewidths=0.35,
    cbar_kws={'label': 'Spearman correlation'},
    ax=ax,
)
ax.set_title('Initial-message feature correlation matrix')
save_figure(fig, 'correlation_heatmap.png')
plt.show()


In [ ]:
# Time series, release-cycle, and topic plots.
monthly_plot = monthly_gold.sort_values('date_period').copy()
monthly_sentiment_column = next((
    column for column in (
        'avg_transformer_expected_sentiment', 'avg_vader_compound'
    ) if column in monthly_plot.columns
), None)

if monthly_sentiment_column:
    fig, left_axis = plt.subplots(figsize=(13, 6))
    right_axis = left_axis.twinx()
    left_axis.plot(
        monthly_plot['date_period'],
        monthly_plot[monthly_sentiment_column],
        color=AUBERGINE,
        linewidth=2,
        label='Average sentiment',
    )
    right_axis.fill_between(
        monthly_plot['date_period'],
        monthly_plot['message_count'],
        color=UBUNTU_ORANGE,
        alpha=0.22,
        label='Message volume',
    )
    left_axis.set(title='Monthly sentiment and message volume', ylabel='Average sentiment')
    right_axis.set_ylabel('Messages')
    left_axis.set_xlabel('Month')
    save_figure(fig, 'monthly_sentiment_and_volume.png')
    plt.show()

if release_gold is not None and monthly_sentiment_column in release_gold.columns:
    release_column = 'days_since_release_bucket'
    fig, ax = plt.subplots(figsize=(11, 5.5))
    sns.barplot(
        data=release_gold,
        x=release_column,
        y=monthly_sentiment_column,
        color=UBUNTU_ORANGE,
        ax=ax,
    )
    ax.tick_params(axis='x', rotation=35)
    ax.set(title='Sentiment across the Ubuntu release cycle', xlabel='Days since release', ylabel='Average sentiment')
    save_figure(fig, 'sentiment_by_release_cycle.png')
    plt.show()

topic_response_summary = pd.DataFrame()
if 'topic_label' in first_messages.columns:
    topic_response_summary = (
        first_messages.dropna(subset=['topic_label'])
        .groupby('topic_label', observed=True)
        .agg(conversations=('conversation_id', 'size'), answer_rate=('was_answered', 'mean'))
        .query('conversations >= 25')
        .nlargest(15, 'conversations')
        .sort_values('answer_rate')
        .reset_index()
    )
    if not topic_response_summary.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.barplot(
            data=topic_response_summary,
            x='answer_rate',
            y='topic_label',
            hue='conversations',
            palette='flare',
            legend=False,
            ax=ax,
        )
        ax.set(title='Conversation answer rate by topic', xlabel='Answer rate', ylabel='Topic')
        save_figure(fig, 'answer_rate_by_topic.png')
        plt.show()

print('Saved figures:')
for figure_path in saved_figures:
    print(' -', figure_path)


---
Step 10 interpretation notes

The figures intentionally separate message volume from average sentiment,
retain neutral-aware transformer expectation when available, and plot only
complete conversation outcomes. The bounded plotting sample affects rendering
cost, while monthly and release-cycle charts use validated gold aggregates.

---


---
Step 11: Machine Learning

The predictive task is whether an initial message receives a reply from a
different sender. This is more useful than predicting a sentiment label from
the same sentiment features, and it avoids inventing a supervised target.

Only information available when the initial message was posted is eligible.
Conversation message count, duration, response gap, later-user activity, raw
usernames, and all later messages are excluded to prevent leakage.

---


In [ ]:
# Build the leakage-safe initial-message model table.
ML_CONVERSATION_LIMIT = 100_000
ml_frame = first_messages.copy()
if len(ml_frame) > ML_CONVERSATION_LIMIT:
    ml_frame = ml_frame.sample(ML_CONVERSATION_LIMIT, random_state=ANALYSIS_SEED)

ml_frame['initial_hour'] = pd.to_datetime(
    ml_frame['date'], errors='coerce', utc=True
).dt.hour
ml_frame['initial_day_of_week'] = pd.to_datetime(
    ml_frame['date'], errors='coerce', utc=True
).dt.day_name()

numeric_feature_candidates = [
    'text_length', 'word_count', 'is_op', 'initial_hour',
    'email_count', 'domain_count', 'phone_count', 'ipv4_count',
    'ipv6_count', 'menu_path_count', 'keyboard_shortcut_count',
    'tech_lexicon_match_count', 'slang_match_count', 'glued_match_count',
    'vader_compound', 'transformer_expected_sentiment',
    'transformer_normalized_entropy', 'nlp_lexical_diversity',
    'nlp_sentence_count', 'nlp_avg_sentence_tokens',
    'nlp_negation_count', 'nlp_punctuation_count', 'noun_ratio',
    'proper_noun_ratio', 'verb_ratio', 'adjective_ratio', 'adverb_ratio',
    'named_entity_count', 'topic_score', 'topic_entropy',
]
categorical_feature_candidates = [
    'initial_day_of_week', 'days_since_release_bucket',
    'days_until_release_bucket', 'topic_label',
]

numeric_features = [
    column for column in numeric_feature_candidates
    if column in ml_frame.columns
    and ml_frame[column].notna().any()
    and ml_frame[column].nunique(dropna=True) > 1
]
categorical_features = [
    column for column in categorical_feature_candidates
    if column in ml_frame.columns
    and ml_frame[column].notna().any()
    and ml_frame[column].nunique(dropna=True) > 1
]
model_features = numeric_features + categorical_features
if not model_features:
    raise ValueError('No eligible initial-message model features are available')

model_df = ml_frame[model_features + ['was_answered']].copy()
model_df['was_answered'] = model_df['was_answered'].astype('int8')
if model_df['was_answered'].nunique() != 2:
    raise ValueError('Response model requires both answered and unanswered conversations')

print(f'Model rows: {len(model_df):,}')
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)
print(model_df['was_answered'].value_counts(normalize=True).rename('proportion'))


In [ ]:
# Train/validation/test split and model selection against a dummy baseline.
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, f1_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = model_df[model_features]
y = model_df['was_answered']
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=ANALYSIS_SEED, stratify=y
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    random_state=ANALYSIS_SEED,
    stratify=y_train_val,
)

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore', min_frequency=20)),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])

ml_workers = min(workers_for(len(model_df)), 4)
candidate_models = {
    'Dummy baseline': DummyClassifier(strategy='prior'),
    'Logistic regression': LogisticRegression(
        max_iter=1_000, class_weight='balanced', random_state=ANALYSIS_SEED
    ),
    'Random forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=18,
        min_samples_leaf=5,
        class_weight='balanced_subsample',
        random_state=ANALYSIS_SEED,
        n_jobs=ml_workers,
    ),
}

def classification_metrics(model_name, fitted_model, features, target):
    probability = fitted_model.predict_proba(features)[:, 1]
    prediction = (probability >= 0.50).astype('int8')
    return {
        'model': model_name,
        'roc_auc': roc_auc_score(target, probability),
        'average_precision': average_precision_score(target, probability),
        'balanced_accuracy': balanced_accuracy_score(target, prediction),
        'f1': f1_score(target, prediction, zero_division=0),
    }

candidate_pipelines = {}
selection_rows = []
for model_name, estimator in candidate_models.items():
    candidate = Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('model', estimator),
    ])
    candidate.fit(X_train, y_train)
    candidate_pipelines[model_name] = candidate
    selection_rows.append(
        classification_metrics(
            model_name, candidate, X_validation, y_validation
        )
    )

model_selection_results = pd.DataFrame(selection_rows).sort_values(
    'average_precision', ascending=False
)
display(model_selection_results.round(4))


In [ ]:
# Tune the best real model on training data only, select the threshold on the
# validation split, then evaluate once on the untouched test split.
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import RandomizedSearchCV

real_model_names = ['Logistic regression', 'Random forest']
selected_model_name = (
    model_selection_results[model_selection_results['model'].isin(real_model_names)]
    .iloc[0]['model']
)
selected_pipeline = clone(candidate_pipelines[selected_model_name])

if selected_model_name == 'Logistic regression':
    parameter_distributions = {
        'model__C': np.logspace(-2, 2, 20),
        'model__class_weight': [None, 'balanced'],
    }
else:
    selected_pipeline.set_params(model__n_jobs=1)
    parameter_distributions = {
        'model__n_estimators': [100, 150, 250],
        'model__max_depth': [10, 18, 26, None],
        'model__min_samples_leaf': [2, 5, 10, 20],
        'model__max_features': ['sqrt', 0.50, 0.75],
        'model__class_weight': ['balanced', 'balanced_subsample'],
    }

tuning_search = RandomizedSearchCV(
    selected_pipeline,
    param_distributions=parameter_distributions,
    n_iter=8,
    scoring='average_precision',
    cv=3,
    random_state=ANALYSIS_SEED,
    n_jobs=ml_workers,
    refit=True,
    return_train_score=False,
)
tuning_search.fit(X_train, y_train)

validation_probability = tuning_search.best_estimator_.predict_proba(X_validation)[:, 1]
precision, recall, thresholds = precision_recall_curve(
    y_validation, validation_probability
)
if len(thresholds):
    threshold_f1 = (
        2 * precision[:-1] * recall[:-1]
        / np.maximum(precision[:-1] + recall[:-1], np.finfo(float).eps)
    )
    selected_threshold = float(thresholds[int(np.nanargmax(threshold_f1))])
else:
    selected_threshold = 0.50

final_model = clone(tuning_search.best_estimator_)
final_model.fit(X_train_val, y_train_val)
test_probability = final_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= selected_threshold).astype('int8')

final_test_metrics = pd.DataFrame([{
    'model': selected_model_name,
    'threshold': selected_threshold,
    'roc_auc': roc_auc_score(y_test, test_probability),
    'average_precision': average_precision_score(y_test, test_probability),
    'balanced_accuracy': balanced_accuracy_score(y_test, test_prediction),
    'f1': f1_score(y_test, test_prediction, zero_division=0),
    'test_rows': len(y_test),
}])

print('Selected model:', selected_model_name)
print('Best parameters:', tuning_search.best_params_)
display(final_test_metrics.round(4))


In [ ]:
# Held-out diagnostics, explainability, and reproducible model export.
import json
from datetime import datetime, timezone

import joblib
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_prediction, cmap='Purples', colorbar=False, ax=axes[0]
)
axes[0].set_title('Held-out confusion matrix')
RocCurveDisplay.from_predictions(
    y_test, test_probability, curve_kwargs={'color': AUBERGINE}, ax=axes[1]
)
axes[1].set_title('Held-out ROC curve')
PrecisionRecallDisplay.from_predictions(
    y_test, test_probability, curve_kwargs={'color': UBUNTU_ORANGE}, ax=axes[2]
)
axes[2].set_title('Held-out precision-recall curve')
save_figure(fig, 'response_model_diagnostics.png')
plt.show()

transformed_feature_names = final_model.named_steps[
    'preprocessor'
].get_feature_names_out()
fitted_estimator = final_model.named_steps['model']
if hasattr(fitted_estimator, 'coef_'):
    importance_values = fitted_estimator.coef_[0]
    importance_kind = 'signed coefficient'
else:
    importance_values = fitted_estimator.feature_importances_
    importance_kind = 'feature importance'

feature_importance = pd.DataFrame({
    'feature': transformed_feature_names,
    'importance': importance_values,
})
feature_importance['absolute_importance'] = feature_importance['importance'].abs()
feature_importance = feature_importance.sort_values(
    'absolute_importance', ascending=False
)

top_features = feature_importance.head(20).sort_values('importance')
fig, ax = plt.subplots(figsize=(10, 7))
colors = [AUBERGINE if value < 0 else UBUNTU_ORANGE for value in top_features['importance']]
ax.barh(top_features['feature'], top_features['importance'], color=colors)
ax.set(title=f'Top response-model {importance_kind}s', xlabel=importance_kind, ylabel='')
save_figure(fig, 'response_model_feature_importance.png')
plt.show()

MODEL_DIR = OUTPUT_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / 'conversation_response_model.joblib'
model_metadata_path = MODEL_DIR / 'conversation_response_model.json'
joblib.dump(final_model, model_path)

model_metadata = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'target': 'was_answered_by_a_different_sender',
    'selected_model': selected_model_name,
    'selected_threshold': selected_threshold,
    'random_seed': ANALYSIS_SEED,
    'training_rows': len(X_train_val),
    'test_rows': len(X_test),
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'best_parameters': tuning_search.best_params_,
    'test_metrics': final_test_metrics.iloc[0].to_dict(),
    'excluded_as_leakage': [
        'conversation_message_count', 'conversation_duration_mins',
        'response_gap_mins_between_speakers', 'from', 'to',
    ],
}
model_metadata_path.write_text(
    json.dumps(model_metadata, indent=2, default=str), encoding='utf-8'
)
print('Saved model:', model_path)
print('Saved model metadata:', model_metadata_path)


---
Step 11 interpretation notes

- The dummy model establishes the class-prevalence baseline.
- Logistic regression supplies an interpretable linear benchmark.
- Random forest captures nonlinear interactions without requiring the response
  target to be continuous or normally distributed.
- Model selection uses validation average precision; hyperparameters are tuned
  only on training data; the final metrics come from one untouched test split.
- Feature importance is predictive, not causal. It should guide investigation
  and support-message triage, not claim that editing one word guarantees a reply.

---


---
Step 12: Evaluation

This final notebook step evaluates the selected response model on the untouched
test split, compares it with the class-prevalence baseline, and translates the
result into a bounded operational triage scenario. It also records limitations,
next steps, and reproducibility documentation.

- Model performance metrics
- Business impact assessment
- Next steps
- Documentation

---


In [ ]:
# Evaluate held-out performance, baseline lift, and a bounded triage scenario.
from sklearn.metrics import confusion_matrix, precision_score, recall_score

test_prevalence = float(y_test.mean())
test_precision = precision_score(y_test, test_prediction, zero_division=0)
test_recall = recall_score(y_test, test_prediction, zero_division=0)
tn, fp, fn, tp = confusion_matrix(
    y_test, test_prediction, labels=[0, 1]
).ravel()
specificity = tn / max(tn + fp, 1)
negative_predictive_value = tn / max(tn + fn, 1)
average_precision_lift = (
    float(final_test_metrics.iloc[0]['average_precision'])
    / max(test_prevalence, np.finfo(float).eps)
)

evaluation_metrics = pd.DataFrame([
    {'metric': 'ROC AUC', 'value': final_test_metrics.iloc[0]['roc_auc']},
    {'metric': 'Average precision', 'value': final_test_metrics.iloc[0]['average_precision']},
    {'metric': 'Average-precision baseline', 'value': test_prevalence},
    {'metric': 'Average-precision lift', 'value': average_precision_lift},
    {'metric': 'Balanced accuracy', 'value': final_test_metrics.iloc[0]['balanced_accuracy']},
    {'metric': 'F1', 'value': final_test_metrics.iloc[0]['f1']},
    {'metric': 'Precision for answered', 'value': test_precision},
    {'metric': 'Recall for answered', 'value': test_recall},
    {'metric': 'Recall for unanswered', 'value': specificity},
])

triage_candidates = int((test_prediction == 0).sum())
business_impact_assessment = pd.DataFrame([{
    'scenario': 'Review conversations predicted to remain unanswered',
    'decision_threshold': selected_threshold,
    'held_out_conversations': len(y_test),
    'triage_candidates': triage_candidates,
    'triage_share': triage_candidates / max(len(y_test), 1),
    'actually_unanswered_in_triage': int(tn),
    'triage_precision_for_unanswered': negative_predictive_value,
    'unanswered_conversations_captured': specificity,
    'answered_conversations_in_triage': int(fn),
}])

display(evaluation_metrics.round(4))
display(business_impact_assessment.round(4))


---
Step 12 interpretation, limitations, and next steps

The operational assessment is deliberately expressed as held-out counts and
rates, not invented savings or causal impact. A predicted-unanswered conversation
is a review candidate, never a reason to suppress, deprioritize, or auto-close it.

Before production use:

1. Re-evaluate on a later time window to measure temporal generalization.
2. Calibrate probabilities and choose a threshold against an explicit review budget.
3. Monitor class prevalence, feature drift, calibration, and subgroup performance
   where appropriate non-sensitive metadata is available.
4. Validate whether intervention actually improves answer rates with a prospective
   experiment; predictive importance is not causal evidence.
5. Retrain only from versioned, validated silver/gold inputs and keep the dummy
   baseline in every evaluation.

---


In [ ]:
# Save the evaluation evidence and a human-readable model card.
EVALUATION_DIR = OUTPUT_DIR / 'evaluation'
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
evaluation_metrics_path = EVALUATION_DIR / 'model_performance_metrics.csv'
business_impact_path = EVALUATION_DIR / 'business_impact_assessment.csv'
model_card_path = EVALUATION_DIR / 'conversation_response_model_card.md'
evaluation_manifest_path = EVALUATION_DIR / 'evaluation_manifest.json'

evaluation_metrics.to_csv(evaluation_metrics_path, index=False)
business_impact_assessment.to_csv(business_impact_path, index=False)

model_card_lines = [
    '# Conversation Response Model Card',
    '',
    '## Intended use',
    'Prioritize initial Ubuntu support messages for human review when they may remain unanswered.',
    '',
    '## Out-of-scope uses',
    'Do not use the score to auto-close messages, evaluate individual users, or claim causal impact.',
    '',
    '## Evaluation design',
    f'- Untouched held-out rows: {len(y_test):,}',
    f'- Random seed: {ANALYSIS_SEED}',
    f'- Selected model: {selected_model_name}',
    f'- Decision threshold: {selected_threshold:.6f}',
    f'- ROC AUC: {float(final_test_metrics.iloc[0]["roc_auc"]):.4f}',
    f'- Average precision: {float(final_test_metrics.iloc[0]["average_precision"]):.4f}',
    f'- Average-precision baseline: {test_prevalence:.4f}',
    '',
    '## Limitations',
    '- Historical prediction does not establish that an intervention causes a reply.',
    '- The model must be re-evaluated across time and monitored for drift.',
    '- No fairness claim is made where suitable evaluation metadata is unavailable.',
    '- Raw text and usernames are excluded from the dashboard-facing bundle.',
    '',
    '## Reproduction',
    'Use the pinned repository environment and rerun notebook Steps 7-12 from validated inputs.',
]
model_card_path.write_text('\n'.join(model_card_lines) + '\n', encoding='utf-8')

evaluation_manifest = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'analysis_seed': ANALYSIS_SEED,
    'analysis_scope': analysis_scope,
    'held_out_rows': len(y_test),
    'model_path': str(model_path),
    'model_metadata_path': str(model_metadata_path),
    'performance_metrics_path': str(evaluation_metrics_path),
    'business_impact_path': str(business_impact_path),
    'model_card_path': str(model_card_path),
}
evaluation_manifest_path.write_text(
    json.dumps(evaluation_manifest, indent=2, default=str), encoding='utf-8'
)
print('Step 12 evaluation documentation:')
for output_path in (
    evaluation_metrics_path, business_impact_path, model_card_path,
    evaluation_manifest_path,
):
    print(' -', output_path)


---
Steps 7-12 complete

The notebook now produces a validated message-level silver artifact,
conversation/time/release gold summaries, corrected statistical tests,
publication-ready figures, a leakage-safe response model, held-out evaluation
evidence, reproducibility documentation, and privacy-bounded dashboard tables.

The final model answers a narrow predictive question: given the initial
message features available at posting time, how likely is a different sender
to reply? It does not measure support quality, user intent, or causal impact.

---
